In [1]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_csv('../data/processed/laptop_eda.csv')

print(df.shape)
print(df.head())
print(df.columns.tolist())

(1020, 28)
   Unnamed: 0                                               Name    Brand  \
0           0  HP Victus 15-fb0157AX Gaming Laptop (AMD Ryzen...       HP   
1           1  Lenovo V15 G4 ‎82YU00W7IN Laptop (AMD Ryzen 3 ...   Lenovo   
2           2  HP 15s-fq5007TU Laptop (12th Gen Core i3/ 8GB/...       HP   
3           3  Samsung Galaxy Book2 Pro 13 Laptop (12th Gen C...  Samsung   
4           4  Tecno Megabook T1 Laptop (11th Gen Core i3/ 8G...    Tecno   

   Price  Rating Processor_brand Processor_name Processor_variant  \
0  50399    4.30             AMD    AMD Ryzen 5             5600H   
1  26690    4.45             AMD    AMD Ryzen 3             7320U   
2  37012    4.65           Intel  Intel Core i3             1215U   
3  69990    4.75           Intel  Intel Core i5             1240P   
4  23990    4.25           Intel  Intel Core i3            1115G4   

   Processor_gen  Core_per_processor  ...  Graphics_brand  \
0            5.0                 6.0  ...         

In [2]:
# Work on a copy, keep original intact
df_fe = df.copy()

# Drop columns we don't need
df_fe.drop(columns=['Unnamed: 0', 'Graphics_integreted_label'], inplace=True)

print(df_fe.shape)
print(df_fe.columns.tolist())

(1020, 26)
['Name', 'Brand', 'Price', 'Rating', 'Processor_brand', 'Processor_name', 'Processor_variant', 'Processor_gen', 'Core_per_processor', 'Low_Power_Cores', 'Energy_Efficient_Units', 'Threads', 'RAM_GB', 'RAM_type', 'Storage_capacity_GB', 'Storage_type', 'Graphics_name', 'Graphics_brand', 'Graphics_integreted', 'Display_size_inches', 'Horizontal_pixel', 'Vertical_pixel', 'ppi', 'Touch_screen', 'Operating_system', 'Log_Price']


## Featuring engineer notes from EDA

Great idea. Here are all the feature engineering notes we documented during EDA:

---

## Feature Engineering Master Notes

### Drop Columns
| Column | Reason |
|---|---|
| `Horizontal_pixel` | Redundant with `ppi` |
| `Vertical_pixel` | Redundant with `ppi` |
| `Rating` | Near zero correlation (-0.04) with price |
| `Processor_name` | Too granular, captured by brand + gen |
| `Processor_variant` | 88 rare categories, too granular |
| `Name` | Identifier, not a feature |

---

### Fix Data Quality Issues
| Column | Issue | Fix |
|---|---|---|
| `Brand` | `ASUS` vs `Asus` | Standardize to title case |
| `Storage_type` | Leading spaces | Strip whitespace |
| `Storage_type` | `NVMe SSD` | Merge into `SSD` |
| `RAM_type` | `LPDDRX4` | Fix to `LPDDR4X` |
| `RAM_type` | `PDDR5X` | Fix to `LPDDR5X` |
| `Graphics_brand` | `Radeon` | Merge into `AMD` |
| `Operating_system` | `Mac 10.15.3\t OS` | Strip tab, merge into `Mac OS` |
| `Operating_system` | `Mac Catalina OS` | Merge into `Mac OS` |
| `Operating_system` | `DOS 3.0 OS` | Merge into `DOS OS` |
| `Operating_system` | `jio` | Map to `Other` |

---

### Group Rare Categories into `Other`
| Column | Categories to Group |
|---|---|
| `Brand` | All brands with < 10 samples |
| `Processor_brand` | `Qualcomm`, `Microsoft`, `HiSilicon` |
| `RAM_type` | `DDR3`, `LPDDR3`, `DDR6` |
| `Graphics_brand` | `Adreno` |
| `Operating_system` | `Ubuntu OS`, `Prime OS`, `Android 11 OS`, `Linux OS` |

---

### Binning
| Column | Bins | Labels |
|---|---|---|
| `RAM_GB` | 2-8, 16, 32+ | Low, Mid, High |
| `Storage_capacity_GB` | 32-256, 512, 1000+ | Low, Mid, High |
| `Processor_gen` | 1-7, 8-11, 12-14 | Legacy, Mid, Modern |
| `Display_size_inches` | ≤13.3, 13.4-14.2, 15.0-16.2, ≥17 | Small, Mid, Large, XLarge |
| `ppi` | <120, 120-150, 150-200, 200-250, 250+ | Low, Standard, High, Very High, Ultra |

---

### Encoding
| Column | Method | Reason |
|---|---|---|
| `Graphics_integreted` | Binary (0/1) | Already True/False |
| `Touch_screen` | Binary (0/1) | Already True/False |
| All other categorical | One-hot or Label encoding | To be decided |

---

### Multicollinearity to Watch
- `Threads` and `Core_per_processor` — keep only one or combine
- `Low_Power_Cores` and `Energy_Efficient_Units` — likely drop after SHAP


## 1. Drop irrelvant columns

In [3]:
# Step 1: Drop irrelevant/redundant columns
df_fe.drop(columns=[
    'Horizontal_pixel',    # redundant with ppi
    'Vertical_pixel',      # redundant with ppi
    'Rating',              # near zero correlation with price
    'Processor_name',      # too granular, captured by brand + gen
    'Processor_variant',   # 88 rare categories, too granular
    'Name'                 # identifier, not a feature
], inplace=True)

print(df_fe.shape)
print(df_fe.columns.tolist())

(1020, 20)
['Brand', 'Price', 'Processor_brand', 'Processor_gen', 'Core_per_processor', 'Low_Power_Cores', 'Energy_Efficient_Units', 'Threads', 'RAM_GB', 'RAM_type', 'Storage_capacity_GB', 'Storage_type', 'Graphics_name', 'Graphics_brand', 'Graphics_integreted', 'Display_size_inches', 'ppi', 'Touch_screen', 'Operating_system', 'Log_Price']


## 2. Fix data quality issues

In [4]:
# 1. Brand — standardize casing
df_fe['Brand'] = df_fe['Brand'].str.strip().str.title()

# 2. Storage_type — strip whitespace and merge NVMe SSD
df_fe['Storage_type'] = df_fe['Storage_type'].str.strip()
df_fe['Storage_type'] = df_fe['Storage_type'].replace({'NVMe SSD': 'SSD'})

# 3. RAM_type — fix typos
df_fe['RAM_type'] = df_fe['RAM_type'].replace({
    'LPDDRX4': 'LPDDR4X',
    'PDDR5X': 'LPDDR5X'
})


# 4. Graphics_brand — merge Radeon into AMD
df_fe['Graphics_brand'] = df_fe['Graphics_brand'].replace({'Radeon': 'AMD'})

# 5. Operating_system — fix naming inconsistencies
df_fe['Operating_system'] = df_fe['Operating_system'].str.strip()
df_fe['Operating_system'] = df_fe['Operating_system'].replace({
    'Mac 10.15.3\t OS': 'Mac OS',
    'Mac Catalina OS': 'Mac OS',
    'DOS 3.0 OS': 'DOS OS',
    'jio': 'Other'
})

# Verify
print(df_fe['Brand'].value_counts().head())
print(df_fe['Storage_type'].value_counts())
print(df_fe['RAM_type'].value_counts())
print(df_fe['Graphics_brand'].value_counts())
print(df_fe['Operating_system'].value_counts())

Brand
Lenovo    217
Hp        213
Asus      158
Dell      116
Msi        97
Name: count, dtype: int64
Storage_type
SSD                1000
Hard Disk            14
Hard Disk & SSD       6
Name: count, dtype: int64
RAM_type
DDR4       544
DDR5       188
LPDDR5     173
LPDDR4X     55
LPDDR5X     39
LPDDR4      16
DDR3         2
Unified      1
DDR6         1
LPDDR3       1
Name: count, dtype: int64
Graphics_brand
Intel     488
NVIDIA    354
AMD       152
Apple      18
ARM         7
Adreno      1
Name: count, dtype: int64
Operating_system
Windows 11 OS    943
Mac OS            20
DOS OS            17
Windows 10 OS     17
Chrome OS         15
Ubuntu OS          3
Prime OS           2
Other              1
Android 11 OS      1
Linux OS           1
Name: count, dtype: int64


In [5]:
df_fe['Brand'] = df_fe['Brand'].replace({
    'Hp': 'HP',
    'Msi': 'MSI',
    'Lg': 'LG',
    'Axl': 'AXL'
})
print(df_fe['Brand'].value_counts().head(10))

Brand
Lenovo     217
HP         213
Asus       158
Dell       116
MSI         97
Acer        69
Samsung     32
Apple       20
Infinix     20
Chuwi        8
Name: count, dtype: int64


## 3. Group rare categories to other

In [6]:
# Rare category grouping for all categorical columns
# Any category with < 10 samples gets grouped into 'Other'

cat_cols = ['Brand', 'Processor_brand', 'RAM_type', 'Graphics_brand', 'Operating_system']

for col in cat_cols:
    counts = df_fe[col].value_counts()
    rare = counts[counts < 10].index
    df_fe[col] = df_fe[col].apply(lambda x: 'Other' if x in rare else x)

# Verify
for col in cat_cols:
    print(f"\n{col}:")
    print(df_fe[col].value_counts())


Brand:
Brand
Lenovo     217
HP         213
Asus       158
Dell       116
MSI         97
Other       78
Acer        69
Samsung     32
Apple       20
Infinix     20
Name: count, dtype: int64

Processor_brand:
Processor_brand
Intel    742
AMD      250
Apple     18
Other     10
Name: count, dtype: int64

RAM_type:
RAM_type
DDR4       544
DDR5       188
LPDDR5     173
LPDDR4X     55
LPDDR5X     39
LPDDR4      16
Other        5
Name: count, dtype: int64

Graphics_brand:
Graphics_brand
Intel     488
NVIDIA    354
AMD       152
Apple      18
Other       8
Name: count, dtype: int64

Operating_system:
Operating_system
Windows 11 OS    943
Mac OS            20
DOS OS            17
Windows 10 OS     17
Chrome OS         15
Other              8
Name: count, dtype: int64


## 4. Extract GPU Tier from Graphics_name before dropping it

In [7]:
import re

def extract_gpu_tier(name):
    name = str(name).upper()
    
    if re.search(r'RTX\s*4\d{2,3}|RTX\s*40', name):
        return 'RTX_4000'
    elif re.search(r'RTX\s*3\d{2,3}|RTX\s*30|RTX\s*A\d+', name):
        return 'RTX_3000'
    elif re.search(r'RTX\s*2\d{2,3}|RTX\s*20', name):
        return 'RTX_2000'
    elif re.search(r'GTX|MX\d+|GEFORCE\s*MX', name):
        return 'GTX_MX'
    elif re.search(r'APPLE|\d+[\s-]CORE\s*GPU', name):
        return 'Apple_Silicon'
    elif re.search(r'INTEGRATED|UHD|IRIS|ARC\s*(A\d+)?|INTEL\s*(HD|GRAPHICS)', name):
        return 'Integrated'
    elif re.search(r'RADEON|AMD\s*GRAPHICS|VEGA|RX\s*\d+', name):
        return 'AMD_Radeon'
    elif re.search(r'ARM|MALI|ADRENO', name):
        return 'Budget_Mobile'
    else:
        return 'Other'

df_fe['GPU_tier'] = df_fe['Graphics_name'].apply(extract_gpu_tier)

print(df_fe['GPU_tier'].value_counts())
print(f"\nRemaining Other:")
print(df_fe[df_fe['GPU_tier'] == 'Other']['Graphics_name'].value_counts())

GPU_tier
Integrated       496
RTX_4000         157
AMD_Radeon       141
RTX_3000         103
RTX_2000          59
GTX_MX            32
Apple_Silicon     18
Budget_Mobile      8
Other              6
Name: count, dtype: int64

Remaining Other:
Graphics_name
Unknown                   2
NVIDIA                    1
AMD Raedon Graphics       1
NVIDIA RTX NVIDIA T550    1
‎NVIDIA Quadro T600       1
Name: count, dtype: int64


In [8]:
df_fe.drop(columns=['Graphics_name'], inplace=True)

print(df_fe.shape)
print(df_fe.columns.tolist())

(1020, 20)
['Brand', 'Price', 'Processor_brand', 'Processor_gen', 'Core_per_processor', 'Low_Power_Cores', 'Energy_Efficient_Units', 'Threads', 'RAM_GB', 'RAM_type', 'Storage_capacity_GB', 'Storage_type', 'Graphics_brand', 'Graphics_integreted', 'Display_size_inches', 'ppi', 'Touch_screen', 'Operating_system', 'Log_Price', 'GPU_tier']


# 5. Binning

In [9]:
#1. RAM_GB
df_fe['RAM_tier'] = pd.cut(df_fe['RAM_GB'], bins=[0, 8, 16, 64], labels=['Low', 'Mid', 'High'])

#2. Storage_capacity_GB
df_fe['Storage_tier'] = pd.cut(df_fe['Storage_capacity_GB'], bins=[0, 256, 512, 4000], labels=['Low', 'Mid', 'High'])

# 3.Processor_gen
df_fe['Processor_gen_tier'] = pd.cut(df_fe['Processor_gen'], bins=[0, 7, 11, 14], labels=['Legacy', 'Mid', 'Modern'])

#4.Display_size_inches
df_fe['Display_tier'] = pd.cut(df_fe['Display_size_inches'], bins=[0, 13.3, 14.2, 16.2, 18], labels=['Small', 'Mid', 'Large', 'XLarge'])

# 5.ppi
df_fe['ppi_tier'] = pd.cut(df_fe['ppi'], bins=[0, 120, 150, 200, 250, 400], labels=['Low', 'Standard', 'High', 'Very_High', 'Ultra'])

#Verify
for col in ['RAM_tier', 'Storage_tier', 'Processor_gen_tier', 'Display_tier', 'ppi_tier']:
    print(f"\n{col}:")
    print(df_fe[col].value_counts())


RAM_tier:
RAM_tier
Mid     545
Low     403
High     72
Name: count, dtype: int64

Storage_tier:
Storage_tier
Mid     702
High    243
Low      75
Name: count, dtype: int64

Processor_gen_tier:
Processor_gen_tier
Modern    641
Legacy    245
Mid       134
Name: count, dtype: int64

Display_tier:
Display_tier
Large     677
Mid       267
Small      46
XLarge     30
Name: count, dtype: int64

ppi_tier:
ppi_tier
Standard     593
High         270
Very_High     84
Low           42
Ultra         31
Name: count, dtype: int64


In [10]:
# Drop original columns replaced by bins
df_fe.drop(columns=['RAM_GB', 'Storage_capacity_GB', 'Processor_gen',  'Display_size_inches', 'ppi'], inplace=True)

print(df_fe.shape)
print(df_fe.columns.tolist())

(1020, 20)
['Brand', 'Price', 'Processor_brand', 'Core_per_processor', 'Low_Power_Cores', 'Energy_Efficient_Units', 'Threads', 'RAM_type', 'Storage_type', 'Graphics_brand', 'Graphics_integreted', 'Touch_screen', 'Operating_system', 'Log_Price', 'GPU_tier', 'RAM_tier', 'Storage_tier', 'Processor_gen_tier', 'Display_tier', 'ppi_tier']


Checking for multicollinearity

In [11]:
# Check correlation between Threads and Core_per_processor
print(f"Correlation between Threads and Core_per_processor: {df_fe['Threads'].corr(df_fe['Core_per_processor']):.2f}")

Correlation between Threads and Core_per_processor: 0.91


In [12]:
print(f"Correlation between Log_Price and Threads: {df_fe['Threads'].corr(df_fe['Log_Price']):.2f}")
print(f"Correlation between Log_Price and Core_per_processor: {df_fe['Core_per_processor'].corr(df_fe['Log_Price']):.2f}")

Correlation between Log_Price and Threads: 0.76
Correlation between Log_Price and Core_per_processor: 0.76


Since the correlation is same and we are having multicollinearity issue, we drop one of the columns

In [13]:
#Drop Core_per_processor — keep Threads
df_fe.drop(columns=['Core_per_processor'], inplace=True)

print(df_fe.shape)
print(df_fe.columns.tolist())

(1020, 19)
['Brand', 'Price', 'Processor_brand', 'Low_Power_Cores', 'Energy_Efficient_Units', 'Threads', 'RAM_type', 'Storage_type', 'Graphics_brand', 'Graphics_integreted', 'Touch_screen', 'Operating_system', 'Log_Price', 'GPU_tier', 'RAM_tier', 'Storage_tier', 'Processor_gen_tier', 'Display_tier', 'ppi_tier']


In [14]:
print(f"Correlation between Log_Price and Low_Power_Cores: {df_fe['Low_Power_Cores'].corr(df_fe['Log_Price']):.2f}")
print(f"Correlation between Log_Price and Energy_Efficient_Units: {df_fe['Energy_Efficient_Units'].corr(df_fe['Log_Price']):.2f}")
print(f"Correlation between Low_Power_Cores and Energy_Efficient_Units: {df_fe['Low_Power_Cores'].corr(df_fe['Energy_Efficient_Units']):.2f}")

Correlation between Log_Price and Low_Power_Cores: 0.24
Correlation between Log_Price and Energy_Efficient_Units: 0.24
Correlation between Low_Power_Cores and Energy_Efficient_Units: 1.00


In [15]:
# Drop both — perfectly correlated with each other and weak price signal
df_fe.drop(columns=['Low_Power_Cores', 'Energy_Efficient_Units'], inplace=True)

print(df_fe.shape)
print(df_fe.columns.tolist())

(1020, 17)
['Brand', 'Price', 'Processor_brand', 'Threads', 'RAM_type', 'Storage_type', 'Graphics_brand', 'Graphics_integreted', 'Touch_screen', 'Operating_system', 'Log_Price', 'GPU_tier', 'RAM_tier', 'Storage_tier', 'Processor_gen_tier', 'Display_tier', 'ppi_tier']


## 6. Adding new features

Planned features to add:

1. is_gaming : Gaming laptops have an effect on the pricing
2. performance_score : combining threads and processor_gen_tier
3. display_quality : ppi x display - screen quality also matters
4. brand_tier — Premium / Mid-High / Mid / Budget according to the mean

In [16]:
gaming_tiers = ['RTX_2000', 'RTX_3000', 'RTX_4000', 'GTX_MX']
df_fe['is_gaming'] = df_fe['GPU_tier'].apply(lambda x: 1 if x in gaming_tiers else 0)

print(df_fe['is_gaming'].value_counts())
print(f"\nMean Log_Price for gaming: {df_fe[df_fe['is_gaming']==1]['Log_Price'].mean():.2f}")
print(f"Mean Log_Price for non-gaming: {df_fe[df_fe['is_gaming']==0]['Log_Price'].mean():.2f}")

is_gaming
0    669
1    351
Name: count, dtype: int64

Mean Log_Price for gaming: 11.51
Mean Log_Price for non-gaming: 10.88


In [17]:
# Calculate mean Log_Price per brand
brand_means = df_fe.groupby('Brand')['Log_Price'].mean().sort_values(ascending=False)
print(brand_means.round(2))

# Define tiers based on mean Log_Price distribution
def assign_brand_tier(mean_price):
    if mean_price >= 11.5:
        return 'Premium'
    elif mean_price >= 11.1:
        return 'Mid_High'
    elif mean_price >= 10.8:
        return 'Mid'
    else:
        return 'Budget'

brand_tier_map = brand_means.apply(assign_brand_tier).to_dict()
df_fe['brand_tier'] = df_fe['Brand'].map(brand_tier_map)

print(f"\nBrand tier mapping:")
print(brand_tier_map)

print(f"\nValue counts:")
print(df_fe['brand_tier'].value_counts())

print(f"\nMean Log_Price by brand_tier:")
print(df_fe.groupby('brand_tier')['Log_Price'].mean().sort_values(ascending=False).round(2))

Brand
Apple      11.95
Samsung    11.59
MSI        11.45
Dell       11.24
HP         11.13
Asus       11.05
Lenovo     10.99
Acer       10.85
Other      10.71
Infinix    10.52
Name: Log_Price, dtype: float64

Brand tier mapping:
{'Apple': 'Premium', 'Samsung': 'Premium', 'MSI': 'Mid_High', 'Dell': 'Mid_High', 'HP': 'Mid_High', 'Asus': 'Mid', 'Lenovo': 'Mid', 'Acer': 'Mid', 'Other': 'Budget', 'Infinix': 'Budget'}

Value counts:
brand_tier
Mid         444
Mid_High    426
Budget       98
Premium      52
Name: count, dtype: int64

Mean Log_Price by brand_tier:
brand_tier
Premium     11.73
Mid_High    11.23
Mid         10.99
Budget      10.67
Name: Log_Price, dtype: float64


We're multiplying the number of threads by the processor generation tier (Legacy=1, Mid=2, Modern=3) to create a single score that represents overall CPU performance.

In [18]:
gen_tier_map = {'Legacy': 1, 'Mid': 2, 'Modern': 3}
df_fe['performance_score'] = df_fe['Threads'] * df_fe['Processor_gen_tier'].map(gen_tier_map).astype(int)

print(df_fe['performance_score'].describe().round(2))
print(f"\nCorrelation with Log_Price: {df_fe['performance_score'].corr(df_fe['Log_Price']):.2f}")

count    1020.00
mean       31.40
std        19.88
min         2.00
25%        16.00
50%        36.00
75%        37.50
max        96.00
Name: performance_score, dtype: float64

Correlation with Log_Price: 0.68


We're multiplying the pixel density tier (ppi) by the screen size tier to create a single score that represents overall display quality — a high-resolution large screen scores higher than a low-resolution small screen.

In [19]:
ppi_tier_map = {'Low': 1, 'Standard': 2, 'High': 3, 'Very_High': 4, 'Ultra': 5}
display_tier_map = {'Small': 1, 'Mid': 2, 'Large': 3, 'XLarge': 4}

df_fe['display_quality'] = (
    df_fe['ppi_tier'].map(ppi_tier_map).astype(int) * 
    df_fe['Display_tier'].map(display_tier_map).astype(int)
)

print(df_fe['display_quality'].describe().round(2))
print(f"\nCorrelation with Log_Price: {df_fe['display_quality'].corr(df_fe['Log_Price']):.2f}")

count    1020.00
mean        6.42
std         2.14
min         2.00
25%         6.00
50%         6.00
75%         6.00
max        20.00
Name: display_quality, dtype: float64

Correlation with Log_Price: 0.65


In [20]:
print(df_fe.shape)
print(df_fe.columns.tolist())

(1020, 21)
['Brand', 'Price', 'Processor_brand', 'Threads', 'RAM_type', 'Storage_type', 'Graphics_brand', 'Graphics_integreted', 'Touch_screen', 'Operating_system', 'Log_Price', 'GPU_tier', 'RAM_tier', 'Storage_tier', 'Processor_gen_tier', 'Display_tier', 'ppi_tier', 'is_gaming', 'brand_tier', 'performance_score', 'display_quality']


## 7.Encoding

In [21]:
# 1. Binary encoding — True/False to 1/0
df_fe['Graphics_integreted'] = df_fe['Graphics_integreted'].astype(int)
df_fe['Touch_screen'] = df_fe['Touch_screen'].astype(int)

# Verify
print(df_fe['Graphics_integreted'].value_counts())
print(df_fe['Touch_screen'].value_counts())

Graphics_integreted
0    774
1    246
Name: count, dtype: int64
Touch_screen
0    904
1    116
Name: count, dtype: int64


In [22]:
from sklearn.preprocessing import LabelEncoder

# Label encoding — for Random Forest & XGBoost
df_label = df_fe.copy()

cat_cols = ['Brand', 'Processor_brand', 'RAM_type', 'Storage_type', 
            'Graphics_brand', 'Operating_system', 'GPU_tier',
            'RAM_tier', 'Storage_tier', 'Processor_gen_tier', 
            'Display_tier', 'ppi_tier', 'brand_tier']

le = LabelEncoder()
for col in cat_cols:
    df_label[col] = le.fit_transform(df_label[col].astype(str))

# One-hot encoding — for Linear Regression
df_encoded = pd.get_dummies(df_fe, columns=cat_cols, drop_first=True)

print(f"Label encoded shape: {df_label.shape}")
print(f"One-hot encoded shape: {df_encoded.shape}")

Label encoded shape: (1020, 21)
One-hot encoded shape: (1020, 61)


In [23]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Label encoded — for Random Forest & XGBoost
df_label.to_csv('../data/processed/laptop_label_encoded.csv', index=False)

# One-hot encoded — for Linear Regression
df_encoded.to_csv('../data/processed/laptop_onehot_encoded.csv', index=False)

print("Both saved!")
print(f"Label encoded shape: {df_label.shape}")
print(f"One-hot encoded shape: {df_encoded.shape}")

Both saved!
Label encoded shape: (1020, 21)
One-hot encoded shape: (1020, 61)
